# ipynb.txt-to-ipynb-converter

Automatically find repository files ending in `.ipynb.txt`, validate them as Jupyter Notebook JSON, upload the corresponding `.ipynb` file to the same directory, verify the uploaded bytes, and only then delete the original `.ipynb.txt`.

## How to use

1. Open this notebook in [Google Colab](https://colab.research.google.com/).
2. Run the notebook cells from top to bottom.
3. When prompted, enter the GitHub repository URL and a GitHub Personal Access Token (PAT). Both prompts use `getpass`, so the values are not echoed.
4. Review the final report and any skipped/failed operations.

No branch input, widgets, buttons, Git installation, paid API, or manual source edits are required.

## Token permissions

Use a PAT with **Contents: read and write** access to the target repository (for a fine-grained token). The token must also be allowed by your organization, if applicable. The app never writes the token to a file or prints it. It is held in memory for the current runtime.

## Safety behavior

- Uses the repository's default branch, discovered through the GitHub REST API.
- Scans the complete Git tree recursively.
- Validates candidate contents as JSON with the expected notebook structure (`nbformat`, `nbformat_minor`, and `cells`).
- Never overwrites an existing `.ipynb` path.
- Checks the uploaded file by downloading it again and comparing raw bytes.
- Deletes the `.ipynb.txt` original only after byte-for-byte verification succeeds.
- If any earlier step fails, the original is retained.
- GitHub API errors, rate limits, permission problems, network errors, and large-file limitations are reported per operation where possible.

> **Important:** Restoring files changes your repository. Review the final report and keep a backup of important work. The process creates a commit for each successful restore/delete sequence; GitHub API operations are not a single atomic transaction. If the upload succeeds but verification or deletion fails, the restored `.ipynb` may remain alongside the original.

In [ ]:
from getpass import getpass
import base64
import json
import re
import time
from urllib.parse import urlparse
import requests

API_ROOT = "https://api.github.com"
API_VERSION = "2022-11-28"
TIMEOUT = (10, 60)
MAX_CONTENTS_API_BYTES = 100 * 1024 * 1024  # GitHub Contents API limit

def api_headers(token):
    return {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {token}",
        "X-GitHub-Api-Version": API_VERSION,
        "User-Agent": "github-notebook-restoration-colab",
    }

def parse_repo_url(value):
    """Accept https://github.com/OWNER/REPO or github.com/OWNER/REPO."""
    value = value.strip()
    if not value:
        raise ValueError("Repository URL cannot be empty.")
    if "://" not in value:
        value = "https://" + value
    parsed = urlparse(value)
    if parsed.hostname not in ("github.com", "www.github.com"):
        raise ValueError("Enter a GitHub repository URL such as https://github.com/OWNER/REPO.")
    parts = [p for p in parsed.path.split("/") if p]
    if len(parts) < 2:
        raise ValueError("Repository URL must include both OWNER and REPOSITORY.")
    owner, repo = parts[0], parts[1]
    if repo.endswith(".git"):
        repo = repo[:-4]
    if not owner or not repo:
        raise ValueError("Could not identify repository owner and name from the URL.")
    return owner, repo

def request_json(session, method, url, **kwargs):
    """Raise readable errors and preserve GitHub's rate-limit details."""
    try:
        response = session.request(method, url, timeout=TIMEOUT, **kwargs)
    except requests.exceptions.Timeout as exc:
        raise RuntimeError(f"Network timeout contacting GitHub: {exc}") from exc
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(f"Network error contacting GitHub: {exc}") from exc

    if response.status_code == 403 and (
        response.headers.get("X-RateLimit-Remaining") == "0"
        or "rate limit" in response.text.lower()
    ):
        reset = response.headers.get("X-RateLimit-Reset")
        detail = f" Rate limit reset epoch: {reset}." if reset else ""
        raise RuntimeError("GitHub API rate limit exceeded." + detail)
    if response.status_code in (401, 403):
        detail = response.text[:500]
        if response.status_code == 401:
            raise RuntimeError("GitHub authentication failed: check the PAT and whether it has expired. " + detail)
        raise RuntimeError("GitHub denied access. Check repository access, token Contents permissions, SSO/organization policy, or rate limits. " + detail)
    if response.status_code == 404:
        raise RuntimeError("GitHub returned 404: repository/path not found, or the token cannot access it.")
    if response.status_code == 409:
        raise RuntimeError("GitHub reports a conflict (409): the branch or repository may have changed. " + response.text[:400])
    if response.status_code == 422:
        raise RuntimeError("GitHub rejected the request (422), often due to validation or file-size limits: " + response.text[:500])
    if response.status_code >= 400:
        raise RuntimeError(f"GitHub API error HTTP {response.status_code}: {response.text[:500]}")
    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise RuntimeError(f"GitHub returned a non-JSON response (HTTP {response.status_code}).") from exc

def validate_notebook(raw):
    """Validate JSON and the basic Jupyter notebook document structure."""
    try:
        doc = json.loads(raw.decode("utf-8-sig"))
    except UnicodeDecodeError as exc:
        raise ValueError("file is not valid UTF-8 JSON.") from exc
    except json.JSONDecodeError as exc:
        raise ValueError(f"invalid JSON at line {exc.lineno}, column {exc.colno}: {exc.msg}.") from exc
    if not isinstance(doc, dict):
        raise ValueError("JSON root must be an object.")
    if not isinstance(doc.get("nbformat"), int) or not isinstance(doc.get("nbformat_minor"), int):
        raise ValueError("missing or invalid integer nbformat/nbformat_minor fields.")
    if not isinstance(doc.get("cells"), list):
        raise ValueError("missing or invalid cells list.")
    for index, cell in enumerate(doc["cells"]):
        if not isinstance(cell, dict):
            raise ValueError(f"cell {index} is not an object.")
        if cell.get("cell_type") not in ("code", "markdown", "raw"):
            raise ValueError(f"cell {index} has an invalid cell_type.")
        if not isinstance(cell.get("source"), (str, list)):
            raise ValueError(f"cell {index} has missing or invalid source.")
    return doc

def download_repo_file(session, owner, repo, path, branch):
    """Get raw file bytes through the GitHub Contents API."""
    encoded_path = "/".join(requests.utils.quote(part, safe="") for part in path.split("/"))
    url = f"{API_ROOT}/repos/{owner}/{repo}/contents/{encoded_path}"
    data = request_json(session, "GET", url, params={"ref": branch},
                        headers={**session.headers, "Accept": "application/vnd.github.raw+json"})
    # Raw media type usually returns bytes, but handle JSON metadata responses too.
    if isinstance(data, dict):
        if data.get("encoding") == "base64" and data.get("content"):
            return base64.b64decode(data["content"])
        raise RuntimeError("GitHub did not return raw file content; file may exceed the Contents API size limit.")
    if isinstance(data, str):
        return data.encode("utf-8")
    # request_json parses JSON; use a raw request for binary-safe exact bytes.
    try:
        r = session.get(url, params={"ref": branch},
                        headers={**session.headers, "Accept": "application/vnd.github.raw"},
                        timeout=TIMEOUT)
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(f"Network error downloading {path}: {exc}") from exc
    if r.status_code >= 400:
        raise RuntimeError(f"Download failed for {path}: HTTP {r.status_code}: {r.text[:400]}")
    return r.content

def get_raw_file(session, owner, repo, path, branch):
    encoded_path = "/".join(requests.utils.quote(part, safe="") for part in path.split("/"))
    url = f"{API_ROOT}/repos/{owner}/{repo}/contents/{encoded_path}"
    try:
        r = session.get(url, params={"ref": branch},
                        headers={**session.headers, "Accept": "application/vnd.github.raw"},
                        timeout=TIMEOUT)
    except requests.exceptions.Timeout as exc:
        raise RuntimeError(f"Network timeout downloading {path}: {exc}") from exc
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(f"Network error downloading {path}: {exc}") from exc
    if r.status_code == 403 and r.headers.get("X-RateLimit-Remaining") == "0":
        raise RuntimeError("GitHub API rate limit exceeded while downloading " + path)
    if r.status_code >= 400:
        raise RuntimeError(f"Download failed for {path}: HTTP {r.status_code}: {r.text[:400]}")
    # A raw media-type response should be the file bytes. Detect unexpected JSON metadata.
    if r.headers.get("Content-Type", "").startswith("application/json"):
        try:
            obj = r.json()
            if obj.get("encoding") == "base64":
                return base64.b64decode(obj.get("content", ""))
        except Exception:
            pass
        raise RuntimeError("GitHub returned metadata instead of raw bytes; file may exceed the Contents API limit.")
    return r.content

def get_file_metadata(session, owner, repo, path, branch):
    encoded_path = "/".join(requests.utils.quote(part, safe="") for part in path.split("/"))
    return request_json(session, "GET", f"{API_ROOT}/repos/{owner}/{repo}/contents/{encoded_path}",
                        params={"ref": branch})

def create_file(session, owner, repo, path, content, branch, message):
    encoded_path = "/".join(requests.utils.quote(part, safe="") for part in path.split("/"))
    url = f"{API_ROOT}/repos/{owner}/{repo}/contents/{encoded_path}"
    payload = {
        "message": message,
        "content": base64.b64encode(content).decode("ascii"),
        "branch": branch,
    }
    return request_json(session, "PUT", url, json=payload)

def delete_file(session, owner, repo, path, branch, sha, message):
    encoded_path = "/".join(requests.utils.quote(part, safe="") for part in path.split("/"))
    url = f"{API_ROOT}/repos/{owner}/{repo}/contents/{encoded_path}"
    return request_json(session, "DELETE", url, json={
        "message": message, "sha": sha, "branch": branch
    })

def main():
    print("GitHub Notebook Restoration")
    repository_url = getpass("Enter your GitHub repository URL: ")
    github_token = getpass("Enter your GitHub Personal Access Token (PAT): ")

    if not github_token.strip():
        print("\nERROR: PAT cannot be empty. No repository changes were made.")
        return
    try:
        owner, repo = parse_repo_url(repository_url)
    except ValueError as exc:
        print(f"\nERROR: {exc}")
        return

    session = requests.Session()
    session.headers.update(api_headers(github_token.strip()))
    # Do not print the token, include it in errors, or write it to disk.
    github_token = None

    report = {
        "repository": f"{owner}/{repo}",
        "default_branch": "Unknown",
        "discovered": 0,
        "restored": [],
        "skipped": [],
        "failed": [],
    }

    try:
        repo_info = request_json(session, "GET", f"{API_ROOT}/repos/{owner}/{repo}")
        default_branch = repo_info.get("default_branch")
        if not default_branch:
            raise RuntimeError("GitHub repository response did not include a default branch.")
        report["default_branch"] = default_branch
        print(f"\nRepository: {owner}/{repo}")
        print(f"Default branch: {default_branch}")

        # Use the recursive Git Trees API to scan all tracked files, including nested folders.
        branch_ref = request_json(session, "GET",
                                  f"{API_ROOT}/repos/{owner}/{repo}/git/ref/heads/{requests.utils.quote(default_branch, safe='')}")
        commit_sha = branch_ref["object"]["sha"]
        commit = request_json(session, "GET",
                              f"{API_ROOT}/repos/{owner}/{repo}/git/commits/{commit_sha}")
        tree_sha = commit["tree"]["sha"]
        tree = request_json(session, "GET",
                            f"{API_ROOT}/repos/{owner}/{repo}/git/trees/{tree_sha}",
                            params={"recursive": "1"})
        if tree.get("truncated"):
            raise RuntimeError("GitHub marked the recursive tree as truncated; a complete scan cannot be guaranteed. No changes were made.")
        matches = [
            item for item in tree.get("tree", [])
            if item.get("type") == "blob" and item.get("path", "").endswith(".ipynb.txt")
        ]
        report["discovered"] = len(matches)
        print(f"Matching .ipynb.txt files discovered: {len(matches)}")

        for item in matches:
            original_path = item["path"]
            restored_path = original_path[:-4]  # remove only the final ".txt"
            print(f"\nProcessing: {original_path}")
            try:
                # Check target path first; never overwrite an existing notebook.
                try:
                    existing = get_file_metadata(session, owner, repo, restored_path, default_branch)
                    if existing:
                        report["skipped"].append((original_path, f"Target already exists: {restored_path}; not overwritten."))
                        print(f"SKIPPED: target already exists: {restored_path}")
                        continue
                except RuntimeError as exc:
                    # A 404 is the normal "target does not exist" case. Distinguish it from other errors.
                    if "returned 404" not in str(exc):
                        raise

                raw = get_raw_file(session, owner, repo, original_path, default_branch)
                validate_notebook(raw)
                if len(raw) > MAX_CONTENTS_API_BYTES:
                    report["skipped"].append((original_path, "File exceeds GitHub Contents API 100 MB limit."))
                    print("SKIPPED: file exceeds GitHub Contents API 100 MB limit.")
                    continue

                # Re-check the target immediately before creation to reduce overwrite/race risk.
                try:
                    get_file_metadata(session, owner, repo, restored_path, default_branch)
                    report["skipped"].append((original_path, f"Target appeared before upload: {restored_path}; not overwritten."))
                    print(f"SKIPPED: target now exists: {restored_path}")
                    continue
                except RuntimeError as exc:
                    if "returned 404" not in str(exc):
                        raise

                create_file(session, owner, repo, restored_path, raw, default_branch,
                            f"Restore notebook extension: {restored_path}")
                # Upload must be confirmed by GitHub, then downloaded and compared byte-for-byte.
                uploaded = get_raw_file(session, owner, repo, restored_path, default_branch)
                if uploaded != raw:
                    report["failed"].append((original_path, "Upload verification failed: downloaded bytes differ; original retained."))
                    print("FAILED: uploaded file differs from original bytes; original retained.")
                    continue

                # Fetch fresh original metadata for its current SHA before deletion.
                original_meta = get_file_metadata(session, owner, repo, original_path, default_branch)
                original_sha = original_meta.get("sha") if isinstance(original_meta, dict) else None
                if not original_sha:
                    report["failed"].append((original_path, "Could not retrieve original file SHA; original retained."))
                    print("FAILED: could not retrieve original file SHA; original retained.")
                    continue

                delete_file(session, owner, repo, original_path, default_branch, original_sha,
                            f"Remove restored text-suffixed notebook: {original_path}")
                report["restored"].append((original_path, restored_path))
                print(f"RESTORED: {original_path} -> {restored_path} (upload verified; original deletion confirmed)")
            except Exception as exc:
                message = str(exc)
                report["failed"].append((original_path, message))
                print(f"FAILED: {message}")

    except Exception as exc:
        report["failed"].append(("<repository scan/setup>", str(exc)))
        print(f"\nFATAL ERROR: {exc}")
    finally:
        session.close()

    print("\n" + "=" * 60)
    print("FINAL REPORT")
    print("=" * 60)
    print(f"Repository: {report['repository']}")
    print(f"Default branch: {report['default_branch']}")
    print(f"Matching files discovered: {report['discovered']}")
    print(f"Notebooks successfully restored: {len(report['restored'])}")
    print(f"Files skipped: {len(report['skipped'])}")
    print(f"Failed operations: {len(report['failed'])}")

    if report["restored"]:
        print("\nSuccessful restorations:")
        for original, restored in report["restored"]:
            print(f"  {original} -> {restored}")
    if report["skipped"]:
        print("\nSkipped files:")
        for path, reason in report["skipped"]:
            print(f"  {path}: {reason}")
    if report["failed"]:
        print("\nFailures:")
        for path, reason in report["failed"]:
            print(f"  {path}: {reason}")
    if report["discovered"] == 0:
        print("\nNo tracked files ending in .ipynb.txt were found on the default branch.")
    print("\nFinished. Check the report above before rerunning.")

main()

GitHub Notebook Restoration
Enter your GitHub repository URL: ··········
Enter your GitHub Personal Access Token (PAT): ··········

Repository: aviroopmandal/portfolio
Default branch: main
Matching .ipynb.txt files discovered: 20

Processing: beeai.ipynb.txt
RESTORED: beeai.ipynb.txt -> beeai.ipynb (upload verified; original deletion confirmed)

Processing: business-intelligence.ipynb.txt
RESTORED: business-intelligence.ipynb.txt -> business-intelligence.ipynb (upload verified; original deletion confirmed)

Processing: crewai.ipynb.txt
RESTORED: crewai.ipynb.txt -> crewai.ipynb (upload verified; original deletion confirmed)

Processing: cut-and-keep.ipynb.txt
RESTORED: cut-and-keep.ipynb.txt -> cut-and-keep.ipynb (upload verified; original deletion confirmed)

Processing: faiss-with-hnsw.ipynb.txt
RESTORED: faiss-with-hnsw.ipynb.txt -> faiss-with-hnsw.ipynb (upload verified; original deletion confirmed)

Processing: feels-like-temperature-predictor.ipynb.txt
RESTORED: feels-like-temper